# 📘 Bài 6 — Xử lý lỗi & viết test

**Tuần 4** — Bài này quyết định bạn viết code *chạy được* hay code *tin cậy được*.

## 1. try / except

In [ ]:
def chia(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("  -> bắt được: chia cho 0")
        return None
    finally:
        print("  -> finally luôn chạy")


print(chia(10, 2))
print(chia(10, 0))

In [ ]:
# Bắt nhiều loại lỗi, và lấy thông tin từ đối tượng lỗi
import json


def doc_cau_hinh(noi_dung):
    try:
        return json.loads(noi_dung)
    except json.JSONDecodeError as e:
        print(f"  JSON hỏng tại dòng {e.lineno}, cột {e.colno}: {e.msg}")
        return {}


print(doc_cau_hinh('{"a": 1}'))
print(doc_cau_hinh("{hỏng"))

## 2. Ba quy tắc

In [ ]:
# ❌ QUY TẮC 1 vi phạm: bắt tất cả -> nuốt luôn lỗi lập trình của chính mình
def xau(nums):
    try:
        return sum(nums) / len(nusm)     # gõ sai tên biến!
    except Exception:
        return 0


print(xau([1, 2, 3]))        # trả về 0, bạn tưởng dữ liệu sai
                             # thực ra là bug gõ nhầm, không bao giờ tìm ra

In [ ]:
# ✅ Bắt đúng loại -> lỗi thật sự vẫn nổi lên
def tot(nums):
    try:
        return sum(nums) / len(nums)
    except ZeroDivisionError:
        return 0


print(tot([1, 2, 3]))
print(tot([]))

| Quy tắc | Nội dung |
|---|---|
| **1** | Bắt lỗi **cụ thể**, không bắt `Exception` |
| **2** | Không bao giờ `except ...: pass` |
| **3** | Chỉ bắt khi bạn **xử lý được**; không xử lý được thì để nó crash — traceback gốc nhiều thông tin hơn |

## 3. Tự ném lỗi — fail fast

In [ ]:
# @loi-co-y  <- ô này CỐ Ý lỗi, dùng để bạn xem thông báo lỗi
def tinh_lai(von, lai_suat, nam):
    if von <= 0:
        raise ValueError(f"Von phai > 0, nhan duoc: {von}")
    if lai_suat < 0:
        raise ValueError(f"Lai suat khong duoc am, nhan duoc: {lai_suat}")
    return von * (1 + lai_suat) ** nam


print(tinh_lai(1000, 0.1, 2))
tinh_lai(-1000, 0.1, 2)

**Thông báo lỗi tốt trả lời được 2 câu hỏi:**
1. Cái gì sai? → *"Von phai > 0"*
2. Giá trị nào gây ra? → *"nhan duoc: -1000"*

Không kiểm tra thì `tinh_lai(-1000, 0.1, 5)` trả về `-1610.51` — một con số vô nghĩa lan xuống tận cuối hệ thống, và bạn mất cả buổi truy nguồn.

**Chọn loại lỗi:** `ValueError` (đúng kiểu, sai giá trị) · `TypeError` (sai kiểu) · `KeyError` (thiếu khoá).

## 4. Loại lỗi riêng

In [ ]:
class LoiSoDuKhongDu(Exception):
    """Nem ra khi rut tien nhieu hon so du."""


def rut(so_du, so_tien):
    if so_tien > so_du:
        raise LoiSoDuKhongDu(f"Can {so_tien:,.0f} nhung chi co {so_du:,.0f}")
    return so_du - so_tien


try:
    rut(100, 500)
except LoiSoDuKhongDu as e:
    print("Xử lý riêng tình huống này:", e)

> Lợi ích: người gọi bắt **đúng loại** thay vì phải đọc nội dung chuỗi để phân biệt. Chuỗi thì đổi lúc nào cũng được, còn loại lỗi là một hợp đồng ổn định.

## 5. Viết test

In [ ]:
def chia_an_toan(a, b):
    if b == 0:
        return None
    return a / b


# Test thủ công bằng assert — cách đơn giản nhất
assert chia_an_toan(10, 2) == 5.0
assert chia_an_toan(10, 0) is None
assert chia_an_toan(0, 5) == 0.0
print("Tất cả đều đúng.")

### Với pytest

Tạo file `test_something.py`:

```python
import pytest
from module_cua_ban import chia_an_toan


def test_chia_binh_thuong():
    assert chia_an_toan(10, 2) == 5.0


def test_chia_cho_khong():
    assert chia_an_toan(10, 0) is None


def test_nem_loi():
    with pytest.raises(ValueError):
        ham_nao_do(-1)
```

Chạy: `pytest -v`

## 6. Ba trường hợp luôn phải test

In [ ]:
# @loi-co-y  <- ô này CỐ Ý lỗi, dùng để bạn xem thông báo lỗi
def trung_binh(nums):
    if not nums:
        return 0.0
    return sum(nums) / len(nums)


# ① Bình thường
assert trung_binh([1, 2, 3]) == 2.0

# ② Biên: rỗng, một phần tử, số 0, số âm
assert trung_binh([]) == 0.0
assert trung_binh([5]) == 5.0
assert trung_binh([-1, 1]) == 0.0

# ③ Lỗi: đầu vào sai kiểu -> hàm này CHƯA xử lý, sẽ crash
print("Hai nhóm đầu OK. Thử nhóm ba:")
trung_binh("abc")

### Vì sao viết test khi code còn nhỏ?

| Lý do | Giải thích |
|---|---|
| **Nhanh hơn thử tay** | Chạy `pytest` 1 giây, thay vì gõ lại input mỗi lần |
| **Tài liệu sống** | Đọc test biết ngay hàm dùng thế nào, nhận gì, trả gì |
| **Dám sửa code** | Đây là giá trị lớn nhất — refactor mà không sợ hỏng thứ khác |

> 🔗 **Liên hệ tới Phase 07:** *eval* cho LLM chính là test, chỉ khác ở chỗ đầu ra không xác định tuyệt đối nên phải chấm theo tiêu chí thay vì so bằng. Kỹ năng đó bắt đầu từ chính bài học này.

## ✍️ Bài tập

```bash
pytest tests/phase01 -v          # chấm toàn bộ Phase 01
```